In [10]:
import warnings
warnings.filterwarnings("ignore")        # hide noisy library warnings

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Settings used by the rest of the notebook
SEED = 67                                # fixed seed so results are reproducible
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid")         # clean chart style
pd.set_option("display.max_columns", 30) # show all columns when printing tables

In [11]:
df = pd.read_csv("../data/proj_machine_readings.csv", parse_dates=["timestamp"])

print(df.shape)
print("timestamp type:", df["timestamp"].dtype)
df.head()

(100000, 13)
timestamp type: datetime64[us]


,timestamp,machine_id,temperature,vibration,humidity,pressure,energy_consumption,machine_status,anomaly_flag,predicted_remaining_life,failure_type,downtime_risk,maintenance_required
0,2025-01-01 00:00:00,39,NaN,-28.65,79.96,3.73,2.16,1,0,106,Normal,0.0,0
1,2025-01-01 00:01:00,29,68.19,57.28,35.94,3.64,0.69,1,0,320,Normal,0.0,0
2,2025-01-01 00:02:00,15,98.94,50.20,72.06,1.00,2.49,1,1,19,Normal,1.0,1
3,2025-01-01 00:03:00,43,90.91,37.65,30.34,3.15,4.96,1,1,10,Normal,1.0,1
4,2025-01-01 00:04:00,8,72.32,40.69,56.71,2.68,0.63,2,0,65,Vibration Issue,0.0,1


In [12]:
# 1. Column types
print(df.dtypes)

# 2. Missing values in each column
print(df.isna().sum())
print("Temperature missing:", round(df["temperature"].isna().mean() * 100, 1), "%")

# 3. Duplicate rows
print("Duplicates:", df.duplicated().sum())

# 4. Min, max and average of each number column
df.describe()

timestamp                   datetime64[us]
machine_id                           int64
temperature                        float64
vibration                          float64
humidity                           float64
pressure                           float64
energy_consumption                 float64
machine_status                       int64
anomaly_flag                         int64
predicted_remaining_life             int64
failure_type                           str
downtime_risk                      float64
maintenance_required                 int64
dtype: object
timestamp                       0
machine_id                      0
temperature                 14000
vibration                       0
humidity                        0
pressure                        0
energy_consumption              0
machine_status                  0
anomaly_flag                    0
predicted_remaining_life        0
failure_type                    0
downtime_risk                   0
maintenance_require

,timestamp,machine_id,temperature,vibration,humidity,pressure,energy_consumption,machine_status,anomaly_flag,predicted_remaining_life,downtime_risk,maintenance_required
count,100000,100000.000000,86000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,2025-02-04 17:19:30,25.499330,75.034410,43.324232,54.995401,3.000405,2.747064,1.002050,0.089160,234.269160,0.089155,0.196970
min,2025-01-01 00:00:00,1.000000,35.550000,-106.980000,30.000000,1.000000,0.500000,0.000000,0.000000,1.000000,0.000000,0.000000
25%,2025-01-18 08:39:45,13.000000,68.280000,37.220000,42.520000,2.000000,1.630000,1.000000,0.000000,97.000000,0.000000,0.000000
50%,2025-02-04 17:19:30,25.000000,75.090000,48.635000,54.980000,3.010000,2.740000,1.000000,0.000000,230.000000,0.000000,0.000000
75%,2025-02-22 01:59:15,38.000000,81.780000,59.250000,67.500000,4.000000,3.870000,1.000000,0.000000,365.000000,0.000000,0.000000
max,2025-03-11 10:39:00,50.000000,121.940000,113.800000,80.000000,5.000000,5.000000,2.000000,1.000000,499.000000,1.000000,1.000000
std,NaN,14.389439,10.036257,29.134306,14.437960,1.152399,1.297865,0.446193,0.284976,150.063062,0.284961,0.397711


Key findings: The dataset has 100,000 rows. Only temperature has missing values (14%). There are no duplicate rows. machine_id is a category label, not a number, so it is used for grouping only. All sensors stay within realistic ranges except vibration, which contains negative values (minimum -106.98) that are physically impossible. About 19.7% of readings require maintenance, so the classes are imbalanced.

In [13]:
negatives = (df["vibration"] < 0).sum()
print("Negative vibration rows:", negatives)
print("Percentage:", round(negatives / len(df) * 100, 2), "%")

Negative vibration rows: 6703
Percentage: 6.7 %


6.7% of vibration readings are negative, which is impossible. Before deciding how to fix them, we check whether they are linked to machine failure.

In [14]:
df["is_negative"] = df["vibration"] < 0

for col in ["machine_status", "failure_type", "anomaly_flag", "maintenance_required"]:
    print(col)
    print((df.groupby(col)["is_negative"].mean() * 100).round(2))

    # Chi-square: is the % of negatives different between groups, or just luck?
    p = stats.chi2_contingency(pd.crosstab(df[col], df["is_negative"]))[1]
    print("p-value:", round(p, 3))
    print()

machine_status
machine_status
0    6.43
1    6.73
2    6.72
Name: is_negative, dtype: float64
p-value: 0.509

failure_type
failure_type
Electrical Fault    6.90
Normal              6.70
Overheating         6.59
Pressure Drop       7.11
Vibration Issue     6.62
Name: is_negative, dtype: float64
p-value: 0.955

anomaly_flag
anomaly_flag
0    6.71
1    6.64
Name: is_negative, dtype: float64
p-value: 0.82

maintenance_required
maintenance_required
0    6.71
1    6.67
Name: is_negative, dtype: float64
p-value: 0.829



The share of negative vibration readings is about 6.7% regardless of machine status, failure type, anomaly flag or maintenance requirement (chi-square p-values all well above 0.05). Negative vibration is therefore not associated with machine failure. This does not prove the cause, but it supports treating them as random sign errors.

In [16]:
# Size of each vibration value, ignoring the minus sign
df["vibration_abs"] = df["vibration"].abs()

# Group readings by size
size_group = pd.cut(df["vibration_abs"], bins=[0, 20, 40, 60, 80, 120], right=False)

# Count negative vs positive readings in each size group
table = pd.crosstab(size_group, df["is_negative"]).rename(columns={False: "positive", True: "negative"})
table["negative / positive"] = (table["negative"] / table["positive"]).round(3)
display(table)

# Extra check: the very smallest readings
near_zero = df[df["vibration_abs"] < 5]
print("Readings with size below 5:", len(near_zero),
      "-> negative share:", round(near_zero["is_negative"].mean() * 100, 1), "%")

is_negative,positive,negative,negative / positive
vibration_abs,,,
"[0, 20)",2072,185,0.089
"[20, 40)",21302,1485,0.070
"[40, 60)",46389,3351,0.072
"[60, 80)",21397,1531,0.072
"[80, 120)",2137,151,0.071


Readings with size below 5: 122 -> negative share: 23.0 %


The negative-to-positive ratio is about 0.07 at every vibration level, consistent with random sign errors. It is slightly higher for readings very close to zero (about 34 extra rows, possibly values that were naturally near zero), which is too few to change our decision.

In [17]:
print("Lowest vibration_abs:", round(df["vibration_abs"].min(), 2))
print("Negative values left:", (df["vibration_abs"] < 0).sum())
print("Skew before:", round(df["vibration"].skew(), 2))
print("Skew after: ", round(df["vibration_abs"].skew(), 2))

Lowest vibration_abs: 0.02
Negative values left: 0
Skew before: -2.19
Skew after:  0.01


In [18]:
# Keep only rows where temperature is known and NOT above 90,
# so the only possible trigger left is vibration
cool = df[df["temperature"].notna() & (df["temperature"] <= 90)].copy()

cool["sign"] = np.where(cool["vibration"] < 0, "negative", "positive")
cool["size_group"] = pd.cut(cool["vibration_abs"], bins=[0, 20, 40, 50, 60, 80, 120], right=False)

proof = cool.groupby(["sign", "size_group"], observed=True)["anomaly_flag"].agg(rows="count", anomalies="sum", rate="mean")
proof["rate"] = (proof["rate"] * 100).round(1)
display(proof)

# Every negative reading with a size above 80
big_negative = (df["vibration"] < 0) & (df["vibration_abs"] > 80)
print("Negative readings with size above 80:", big_negative.sum())
print("Share flagged as anomaly:", df.loc[big_negative, "anomaly_flag"].mean())

rows  anomalies   rate
sign     size_group                         
negative [0, 20)       139          0    0.0
         [20, 40)     1107          0    0.0
         [40, 50)     1234          0    0.0
         [50, 60)     1265          0    0.0
         [60, 80)     1146          0    0.0
         [80, 120)     110        110  100.0
positive [0, 20)      1669          0    0.0
         [20, 40)    17167          0    0.0
         [40, 50)    18860          1    0.0
         [50, 60)    18579          1    0.0
         [60, 80)    17147          1    0.0
         [80, 120)    1745       1743   99.9

Negative readings with size above 80: 151
Share flagged as anomaly: 1.0


We use vibration_abs instead of imputation. The negatives are random sign errors, so the magnitude is still a valid reading. The anomaly flag treats negative and positive vibration identically: no negative reading with a size up to 80 is flagged, and all 151 with a size above 80 are flagged. Imputation would replace 6,703 recoverable readings with one repeated value and lose information. Only 5 of about 86,000 rows (0.006%) disagree with the 80-threshold rule, probably due to rounding. No rows are deleted.

In [19]:
df["temp_missing"] = df["temperature"].isna().astype(int)

for col in ["machine_status", "anomaly_flag", "maintenance_required", "failure_type", "machine_id"]:
    rate = (df.groupby(col)["temp_missing"].mean() * 100).round(1)
    p = stats.chi2_contingency(pd.crosstab(df[col], df["temp_missing"]))[1]
    print(f"{col}: missing rate from {rate.min()}% to {rate.max()}%, p-value = {p:.3f}")

machine_status: missing rate from 13.8% to 14.0%, p-value = 0.828
anomaly_flag: missing rate from 13.8% to 14.0%, p-value = 0.528
maintenance_required: missing rate from 13.8% to 14.1%, p-value = 0.312
failure_type: missing rate from 12.5% to 14.2%, p-value = 0.412
machine_id: missing rate from 12.6% to 15.7%, p-value = 0.899


In [20]:
rng = np.random.default_rng(SEED)

# Pretend 14% of the KNOWN temperatures are missing, then fill them in and measure the error
known = df[df["temperature"].notna()]
hide = rng.random(len(known)) < 0.14
train, test = known[~hide], known[hide]
truth = test["temperature"]

for name, fill in {"mean": train["temperature"].mean(), "median": train["temperature"].median()}.items():
    error = fill - truth
    print(f"{name:6s} fill = {fill:.2f}   RMSE = {np.sqrt((error ** 2).mean()):.2f}   MAE = {error.abs().mean():.2f}")

print("Spread (std) of temperature:", round(known["temperature"].std(), 2))

hot = truth > 90
print("Hidden readings above 90 °C:", hot.sum(), "| median fill value:", round(train["temperature"].median(), 2),
      "(below 90, so none of them can be recovered)")

mean   fill = 75.05   RMSE = 9.99   MAE = 8.01
median fill = 75.08   RMSE = 9.99   MAE = 8.01
Spread (std) of temperature: 10.04
Hidden readings above 90 °C: 734 | median fill value: 75.08 (below 90, so none of them can be recovered)


Temperature is missing completely at random (all chi-square p-values above 0.05), so imputation is appropriate. Mean and median perform identically because temperature is symmetric. We choose the median (robust to extreme values) plus a temp_missing marker. A constant fill cannot recover readings above 90 °C, so some overheating events are invisible in rows with a missing temperature; this is a limit of the data. In the pipeline, the median will be calculated from the training data only to avoid leaking test information. For now we keep the original NaN values so that plots are not distorted.

In [21]:
df = df.drop(columns=["is_negative"], errors="ignore")
df.columns.tolist()

['timestamp',
 'machine_id',
 'temperature',
 'vibration',
 'humidity',
 'pressure',
 'energy_consumption',
 'machine_status',
 'anomaly_flag',
 'predicted_remaining_life',
 'failure_type',
 'downtime_risk',
 'maintenance_required',
 'vibration_abs',
 'temp_missing']